# Summary

01_quickstart.ipynb

Single scan inference end-to-end. Load a scan, run NeuroFM.predict(), show the output brain health and latent features.

# Install

In [ ]:
!pip install git+https://github.com/rockNroll87q/NeuroFM.git
!pip install templateflow==25.1.2

# Python API

## Imports

In [ ]:
import templateflow.api as tflow
from neurofm import NeuroFM
from neurofm.model import BRAIN_HEALTH_KEYS

## Create template test vol

For the purposes of illustration, we load an MNI test volume. In your own use-case, this can be replaced by a single volume, a directory of volumes, or a `.csv` file with an input column pointing to your volume files.

In [ ]:
print("Fetching MNI152NLin2009cAsym 1mm T1w template from TemplateFlow...")
template_path = tflow.get(
    "MNI152NLin2009cAsym",
    resolution=1,
    desc="brain",
    suffix="T1w",
    extension=".nii.gz",
)

if template_path is None:
    print("TemplateFlow could not fetch the template. Check your internet connection.")

## Run NeuroFM

In [ ]:
model_variant = "neurofm-s" # choose your variant
device = "cpu"

In [ ]:
model = NeuroFM(
    variant=model_variant,
    device=device,
)

In [ ]:
results = model.predict(template_path, outputs=['brain_health', 'latent'])
brain_health = results['brain_health']
latent = results['latent']

### Brain health outputs

In [ ]:
print(f'Output order: {BRAIN_HEALTH_KEYS}')

In [ ]:
print(f'Predicted age: {brain_health[0]} years')
print(f'Predicted sex: {brain_health[1]} (0 == F, 1 == M)')
print(f'Predicted ventricle vol: {brain_health[2]} mm^3') 
print(f'Predicted brain vol: {brain_health[3]} mm^3')

### Latent outputs

Latent embedded representation of the input brain volume. Dimensionality depends on model variant.

In [ ]:
print(latent)

# Script method

## Create template test vol

Create the template test data and save to /tmp/

!python scripts/get_test_data.py -o /tmp/test_data/ --mode template -n 1

## Run inference script to produce output files

In [ ]:
!python scripts/run_inference.py -i /tmp/test_data/ -o /tmp/nfm_results --device cpu --model neurofm-s --outputs brain_health,latent

In [ ]:
!ls /tmp/nfm_results

Show the output summary .csv (brain health data for all volumes)

In [ ]:
import pandas as pd

pd.read_csv('/tmp/nfm_results/results_summary.csv')